In [0]:
WITH all_trips AS (
    -- Stack all three monthly tables into one unified dataset.
    -- UNION ALL keeps every row from every table.
    SELECT * FROM yellow_tripdata_january
    UNION ALL
    SELECT * FROM yellow_tripdata_february
    UNION ALL
    SELECT * FROM yellow_tripdata_march
),

weather_flags AS (
    -- Create a clear rainy/dry flag per calendar day.
    -- COALESCE(WT16, 0) replaces NULL with 0 so the OR condition works safely.
    SELECT
        DATE                                                    AS weather_date,
        PRCP,
        TMAX / 10.0                                             AS temp_max_celsius,
        TMIN / 10.0                                             AS temp_min_celsius,
        CASE
            WHEN PRCP > 25 OR COALESCE(WT16, 0) = 1 THEN 1
            ELSE 0
        END                                                     AS is_rainy_day
    FROM `ny-weather-data`
),

clean_trips AS (
    -- Join each taxi trip to its weather day and remove known bad data.
    -- DATE() strips the time from the pickup timestamp to match the daily weather record.
    SELECT
        t.tpep_pickup_datetime,
        t.passenger_count,
        t.trip_distance,
        t.PULocationID,
        t.DOLocationID,
        t.fare_amount,
        t.total_amount,
        DATE(t.tpep_pickup_datetime)                            AS pickup_date,
        w.is_rainy_day,
        w.PRCP                                                  AS precipitation_tenths_mm,
        w.temp_max_celsius
    FROM all_trips t
    INNER JOIN weather_flags w
        ON DATE(t.tpep_pickup_datetime) = w.weather_date
    WHERE
        t.fare_amount       > 0               -- $0 fares are invalid
        AND t.total_amount  > 0
        AND t.trip_distance > 0               -- Zero-distance = GPS error or cancelled trip
        AND t.passenger_count BETWEEN 1 AND 6 -- 0 and 9+ are data entry errors
)

SELECT
    CASE WHEN is_rainy_day = 1 THEN 'Rainy Day' ELSE 'Dry Day'
    END                                                         AS weather_type,
    COUNT(DISTINCT pickup_date)                                 AS number_of_days,
    COUNT(*)                                                    AS total_trips,
    -- Multiply by 1.0 to force decimal division (SQL integer / integer = integer)
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT pickup_date), 0)     AS avg_trips_per_day,
    ROUND(AVG(fare_amount),     2)                              AS avg_fare_usd,
    ROUND(AVG(trip_distance),   2)                              AS avg_trip_distance_miles,
    ROUND(AVG(passenger_count), 2)                              AS avg_passengers_per_trip
FROM clean_trips
GROUP BY is_rainy_day
ORDER BY is_rainy_day DESC; -- Rainy Day (1) shows first